# Public holidays around the world

This notebook uses **python-holidays** to count how many **public holiday dates** each country observes in a chosen year (national calendars as modeled by the library — not bank-only or employer-specific rules).

**Why Sweden used to look huge:** In Sweden, **every Sunday** is classified as a *helgdag* in certain laws, and the library’s default `Sweden` calendar **includes all Sundays** (~50) plus the usual fixed/moveable days. That is **not** “50 extra paid public holidays” in everyday language. For fairer **cross-country** charts, this notebook counts Sweden with `include_sundays=False` (still ~13 dates in a typical year). You can change that in the code if you want the full legalistic calendar.

You will see:
- **World map** — choropleth of holiday counts (where data exists for ISO codes).
- **Top countries** — who schedules the most public holidays on the calendar.
- **Distribution** — how counts spread across all supported countries.

In [15]:
import holidays
import pandas as pd
import plotly.express as px

try:
    import pycountry

    def country_name(iso2: str) -> str:
        c = pycountry.countries.get(alpha_2=iso2)
        return c.name if c else iso2
except ImportError:
    def country_name(iso2: str) -> str:
        return iso2


def show_fig(fig) -> None:
    """Inline in Jupyter when ipython/nbformat are installed; otherwise open browser."""
    try:
        fig.show()
    except ValueError as err:
        msg = str(err).lower()
        if "nbformat" in msg or "ipython" in msg or "mime type" in msg:
            fig.show(renderer="browser")
            print(
                "(Opened in browser. For inline charts: python -m pip install 'nbformat>=4.2' ipython)"
            )
        else:
            raise


YEAR = 2026  # change to compare other years

In [16]:
def load_country_calendar(code: str, year: int):
    """Sweden defaults in the library to *all Sundays* as holidays; use include_sundays=False for comparability."""
    if code == "SE":
        return holidays.Sweden(years=year, include_sundays=False)
    return holidays.country_holidays(code, years=year)


def build_holiday_count_table(year: int) -> pd.DataFrame:
    """One row per ISO 3166-1 alpha-2 country: number of distinct public holiday dates."""
    rows: list[dict] = []
    for code in sorted(holidays.list_supported_countries()):
        if len(code) != 2:
            continue
        try:
            cal = load_country_calendar(code, year)
            n = len(cal)
        except Exception:
            continue
        iso3 = None
        try:
            import pycountry as _pc

            pc = _pc.countries.get(alpha_2=code)
            if pc:
                iso3 = pc.alpha_3
        except ImportError:
            pass
        rows.append(
            {
                "iso2": code,
                "iso3": iso3,
                "country": country_name(code),
                "public_holidays": n,
            }
        )
    df = pd.DataFrame(rows)
    return df.sort_values("public_holidays", ascending=False).reset_index(drop=True)


df = build_holiday_count_table(YEAR)
print(f"Countries with data: {len(df)}")
df.head(15)

Countries with data: 251


,iso2,iso3,country,public_holidays
0,SE,SWE,Sweden,63
1,NP,NPL,Nepal,32
2,IR,IRN,"Iran, Islamic Republic of",32
3,VA,VAT,Holy See (Vatican City State),29
4,MM,MMR,Myanmar,27
5,AZ,AZE,Azerbaijan,25
6,LK,LKA,Sri Lanka,25
7,TH,THA,Thailand,23
8,AL,ALB,Albania,23
9,CN,CHN,China,22


## Who has the most public holidays?

Higher counts often reflect **more one-off observances**, **regional layers** folded into the national object in some models, or **long lists of commemorative days** — the library’s definition varies by country. The ranking is still useful for **relative** comparison.

In [17]:
top_n = 30
top = df.head(top_n).iloc[::-1]

fig_bar = px.bar(
    top,
    x="public_holidays",
    y="country",
    orientation="h",
    color="public_holidays",
    color_continuous_scale="Turbo",
    title=f"Top {top_n} countries by number of public holiday dates ({YEAR})",
    labels={"public_holidays": "Distinct holiday dates", "country": ""},
)
fig_bar.update_layout(height=900, yaxis=dict(categoryorder="total ascending"), showlegend=False)
show_fig(fig_bar)

(Opened in browser. For inline charts: python -m pip install 'nbformat>=4.2' ipython)


## World map

Darker = more public holiday **dates** in that year (per **python-holidays** national calendar). Missing greys are usually missing or non-mappable ISO codes in Plotly’s world layer.

In [18]:
# Plotly 6+ only allows ISO-3 (not alpha-2) for built-in world choropleths
df_map = df.dropna(subset=["iso3"])
fig_map = px.choropleth(
    df_map,
    locations="iso3",
    color="public_holidays",
    color_continuous_scale="Viridis",
    locationmode="ISO-3",
    hover_name="country",
    hover_data={"iso2": True, "iso3": True, "public_holidays": True},
    title=f"Public holiday dates by country ({YEAR})",
)
fig_map.update_layout(
    height=560,
    geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
    margin=dict(l=0, r=0, t=50, b=0),
)
show_fig(fig_map)

(Opened in browser. For inline charts: python -m pip install 'nbformat>=4.2' ipython)


## Distribution and extremes

Summary statistics help you see whether most countries cluster in a narrow band or spread wide.

In [19]:
summary = df["public_holidays"].describe()
display(summary.to_frame("days"))

winner = df.iloc[0]
lightest = df.iloc[-1]
print(
    f"Most in dataset: {winner['country']} ({winner['iso2']}) — {int(winner['public_holidays'])} dates"
)
print(
    f"Fewest in dataset: {lightest['country']} ({lightest['iso2']}) — {int(lightest['public_holidays'])} dates"
)

,days
count,251.000000
mean,13.876494
std,5.393764
min,0.000000
25%,11.000000
50%,13.000000
75%,15.500000
max,63.000000


Most in dataset: Sweden (SE) — 63 dates
Fewest in dataset: Ukraine (UA) — 0 dates


In [20]:
fig_hist = px.histogram(
    df,
    x="public_holidays",
    nbins=40,
    title=f"How many countries fall into each holiday-count bucket ({YEAR})",
    labels={"public_holidays": "Public holiday dates (bucket)", "count": "Number of countries"},
)
fig_hist.update_layout(bargap=0.05)
show_fig(fig_hist)

(Opened in browser. For inline charts: python -m pip install 'nbformat>=4.2' ipython)


## Compare a few countries you care about

Edit the list below (ISO 3166-1 alpha-2 codes).

In [21]:
pick = ["LT", "PL", "DE", "FR", "GB", "US", "JP", "IN", "BR", "NG"]
sub = df[df["iso2"].isin(pick)].sort_values("public_holidays", ascending=True)

fig_pick = px.bar(
    sub,
    x="public_holidays",
    y="iso2",
    orientation="h",
    text="public_holidays",
    color="public_holidays",
    color_continuous_scale="Blues",
    title=f"Selected countries ({YEAR})",
    labels={"public_holidays": "Holiday dates", "iso2": "Country"},
)
fig_pick.update_traces(textposition="outside")
fig_pick.update_layout(height=420, showlegend=False)
show_fig(fig_pick)

(Opened in browser. For inline charts: python -m pip install 'nbformat>=4.2' ipython)
